# Portfolio Snapshot Report

A simple, investor-facing view: current positions, portfolio value over time, and
performance vs. benchmark. Deliberately minimal -- one chart, one table -- reading
directly from `data/portfolio_snapshot_<name>.csv` (written automatically by
`daily_runner.py` every run, independent of rebalance schedule).

This is NOT a replacement for the trade log or `measure_live_performance()` -- it's a
faster, higher-level view for "how are we doing" without needing to replay every trade.

In [ ]:
# Epic 17: package is pip-installed editable, no sys.path hacking needed
from momentum_trading.execution.live_signal import get_latest_snapshot
from momentum_trading.core.paths import data_dir
import momentum_trading.core.functions_quant_extensions as fnx
import pandas as pd
import matplotlib.pyplot as plt

## 1. Current snapshot

In [2]:
# EDIT: your portfolio name from config.yaml
portfolio_name = "portfolio1"

latest = get_latest_snapshot(portfolio_name)
if latest is None:
    print(f"No snapshot log found yet for '{portfolio_name}'. Run daily_runner.py at least once first.")
else:
    print(f"As of: {latest['date']}")
    print(f"Total value:     ${float(latest['total_value']):,.2f}")
    print(f"Cash:            ${float(latest['cash']):,.2f}")
    print(f"Positions value: ${float(latest['positions_value']):,.2f}")
    print(f"Unrealized P&L:  ${float(latest['unrealized_pnl']):,.2f}")
    print(f"Open positions:  {latest['n_positions']}")
    if latest['positions_detail']:
        print("\nPositions:")
        for p in str(latest['positions_detail']).split(';'):
            print(f"  {p}")

No snapshot log found yet for 'portfolio1'. Run daily_runner.py at least once first.


## 2. Cumulative return vs. benchmark

In [3]:
comparison = fnx.compare_to_benchmark(portfolio_name)
if "error" in comparison:
    print(comparison["error"])
elif comparison["n_periods"] == 0:
    print("Not enough snapshots yet to compute a period return (need at least 2 runs).")
else:
    print(f"As of: {comparison['as_of_date'].date()}  ({comparison['n_periods']} periods)")
    print(f"Portfolio cumulative return: {comparison['portfolio_cumulative_return']:.2%}")
    print(f"Benchmark cumulative return: {comparison['benchmark_cumulative_return']:.2%}")
    print(f"Outperformance:              {comparison['outperformance']:+.2%}")

No snapshot log found at data/portfolio_snapshot_portfolio1.csv


## 3. Portfolio value over time

In [ ]:
snapshot_path = str(data_dir() / f"portfolio_snapshot_{portfolio_name}.csv")
try:
    df = pd.read_csv(snapshot_path, parse_dates=["date"])
    plt.figure(figsize=(10, 5))
    plt.plot(df["date"], df["total_value"], marker="o", label="Portfolio value")
    plt.title(f"{portfolio_name}: Portfolio Value Over Time")
    plt.ylabel("$")
    plt.xlabel("Date")
    plt.legend()
    plt.tight_layout()
    plt.show()
    display(df[["date", "total_value", "cash", "positions_value", "unrealized_pnl", "n_positions"]].tail(10))
except FileNotFoundError:
    print(f"No snapshot file at {snapshot_path} yet.")

---
## 4. Correlation with your OTHER holdings (Epic 6, Story 6.4)

This strategy's internal correlation penalty only looks at correlation AMONG its own
picks -- it has no visibility into anything else you own. A momentum sleeve that looks
well-diversified internally can still be highly correlated with the rest of your net worth.

Fill in your other holdings' periodic returns below (e.g. from your brokerage statements
or another return series) to check this before committing significant capital.

In [ ]:
# EDIT: replace with your actual strategy period returns (from the snapshot log,
# once you have >=2 snapshots) and your other holdings' returns, same frequency/index.
snapshot_path = str(data_dir() / f"portfolio_snapshot_{portfolio_name}.csv")
try:
    snap_df = pd.read_csv(snapshot_path, parse_dates=["date"]).dropna(subset=["portfolio_period_return"])
    strategy_returns = snap_df.set_index("date")["portfolio_period_return"]

    if len(strategy_returns) < 3:
        print("Not enough periods yet for a meaningful correlation check (need several snapshots).")
    else:
        # EDIT: supply your other real holdings' returns here, aligned by date
        other_holdings_returns = {
            # "my_sp500_index_fund": pd.Series({...}),
            # "my_bond_fund": pd.Series({...}),
        }
        if not other_holdings_returns:
            print("No other holdings supplied yet -- edit the cell above to add your real return series.")
        else:
            result = fnx.check_external_correlation(strategy_returns, other_holdings_returns)
            for name, stats in result["per_holding"].items():
                print(f"{name}: correlation={stats['correlation']:.2f} ({stats['n_overlapping_periods']} overlapping periods)")
            for w in result["warnings"]:
                print(f"\n⚠️  {w}")
except FileNotFoundError:
    print(f"No snapshot file at {snapshot_path} yet.")